In [ ]:
# 5 clinical validation 
import os 
import glob
from tqdm import tqdm
import nibabel as nib
import numpy as np
import neuroimage_analysis as na
import matplotlib.pyplot as plt 
import pandas as pd
from scipy.stats import pearsonr
import seaborn as sns
from nilearn.image import resample_to_img
from scipy.ndimage import affine_transform
from scipy.optimize import minimize_scalar

## Analysis 5: Clinical Prediction — Cohort (Leave-one-subject-out sLNM) vs. Control Maps

To assess the clinical utility of sLNM, we compare its predictive performance for TMS response in the Boston TMS cohort against control maps derived from unrelated datasets spanning migraine to epilepsy, as well as the GSP1000 degree map.

**Leave-one-out prediction**
For each left-out subject, the sLNM map is computed from all remaining subjects. Prediction is defined as the spatial correlation between the held-out subject's FC map and the group sLNM map.

**Control map predictions**
We repeat the same procedure using control maps in place of the cohort-specific leave-one-subject-out sLNM map. Because control maps were not derived from the Boston TMS cohort, any predictive performance reflects non-disease-specific features of these maps.

**Controlling for PC signals**
We then compute partial predictions by controlling for PC1, PC2, or PC3 in turn. If the cohort-specific map's prediction improves relative to controls after partialling out a given PC, this supports the notion that the PC drives non-specific prediction signals.

For variables $x$ and $y$ with covariate $z$, the partial correlation controlling for $z$ is:

$$r_{xy.z}=\frac{r_{xy}-r_{xz}r_{yz}}{\sqrt{1-r_{xz}^2}\sqrt{1-r_{yz}^2}}$$

Here, $x$ is TMS response (% BDI reduction), $y$ is the spatial similarity between each subject's TMS-seeded FC map and the predictor map (Fisher-z transformed Pearson's $r$), and $z$ is the spatial similarity between each subject's TMS-seeded FC map and the PC map. The magnitude of $r_{xy}$ reflects the prediction value of the predictor map without controlling for PC. Because each TMS seed may occupy a different position in PC space — which captures the dominant axis of variance in functional connectivity (e.g., transmodal to unimodal gradient in PC1) and may therefore explain variance in TMS response — partialling out this factor isolates prediction that is specific to the predictor map itself.

**Outcome**
Predictive performance is compared by rank-ordering the cohort leave-one-subject-out sLNM map against control maps across both datasets.

In [ ]:
dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
outdir = os.path.join(dir, "results")

brain_template = nib.load(os.path.join(dir, "data/templates/Taylor_NHB_MNI152_T1_2mm_brain_mask_dil.nii.gz"))
brain_mask = brain_template.get_fdata() > 0

pca_files = sorted(glob.glob(os.path.join(dir, "data/pca/pca_voxelwise_*.nii.gz")))
pc1_map = na.nifti_getdata(pca_files[0])
pc2_map = na.nifti_getdata(pca_files[1])
pc3_map = na.nifti_getdata(pca_files[2])

def partial_cor(A, B, C):
    # correlate A, B controlling for C
    r = np.corrcoef([A, B, C])
    r_AB, r_AC, r_BC = r[0,1], r[0,2], r[1,2]
    return (r_AB - r_AC * r_BC) / (np.sqrt(1 - r_AC**2) * np.sqrt(1 - r_BC**2))

def get_slnm(fc_paths, behavior, visualize=True, outdir=None, filename=None):
    """
    Compute sLNM map by correlating FC maps with a behavior vector.
    If visualize=True, saves NIfTI and CIFTI surface files to outdir.
    """
    fc_array = np.hstack([nib.load(f).get_fdata()[brain_mask].reshape(-1, 1) for f in fc_paths]).T
    behavior = np.array(behavior).reshape(-1, 1)
    lnm_map = na.voxel_outcome_correlation(fc_array, behavior)

    if visualize:
        brain_3d = np.zeros_like(brain_template.get_fdata())
        brain_3d[brain_mask] = lnm_map
        lnm_img = nib.Nifti1Image(brain_3d, affine=brain_template.affine, header=brain_template.header)
        out_path = os.path.join(outdir, f'{filename}.nii.gz')
        nib.save(lnm_img, out_path)
        print(f'Saved: {out_path}')
        na.nifti_to_cifti(nii_path=out_path, outdir=outdir)

    return lnm_map


def loo_prediction(fc_file, behavior):

    # generate LOO sLNM map for each held-out subject
    
    similarity_r_map = []
    alignment_pc1 = []
    alignment_pc2 = []
    alignment_pc3 = []

    for i in tqdm(range(len(fc_file)), desc='LOO analysis'):
        files = fc_file.copy()
        train_files = files[:i] + files[i+1:]

        behavior_all = behavior.copy()
        behavior_train = behavior_all[:i] + behavior_all[i+1:]

        loo_circuit = get_slnm(
            fc_paths=train_files,
            behavior=behavior_train,
            visualize=False,
            outdir=None,
            filename=None
        )

        loo_subject_fc = na.nifti_getdata(fc_file[i])

        valid = (loo_subject_fc != 0) & (~np.isnan(loo_subject_fc)) & (loo_circuit != 0) & (~np.isnan(loo_circuit)) & (pc1_map != 0) & (~np.isnan(pc1_map)) & (pc2_map != 0) & (~np.isnan(pc2_map)) & (pc3_map != 0) & (~np.isnan(pc3_map))

        loo_similarity = pearsonr(loo_subject_fc[valid], loo_circuit[valid]).statistic
        similarity_r_map.append(loo_similarity)

        alignment_pc1.append(np.corrcoef(loo_subject_fc[valid], pc1_map[valid])[0, 1])
        alignment_pc2.append(np.corrcoef(loo_subject_fc[valid], pc2_map[valid])[0, 1])
        alignment_pc3.append(np.corrcoef(loo_subject_fc[valid], pc3_map[valid])[0, 1])

    x = np.array(behavior)
    y = np.arctanh(np.array(similarity_r_map))
    z1 = np.arctanh(np.array(alignment_pc1))
    z2 = np.arctanh(np.array(alignment_pc2))
    z3 = np.arctanh(np.array(alignment_pc3))

    r_full = pearsonr(x, y)[0]
    r_partial_pc1 = partial_cor(x, y, z1)
    r_partial_pc2 = partial_cor(x, y, z2)
    r_partial_pc3 = partial_cor(x, y, z3)

    print(f'LOO prediction (full):        r = {r_full:.3f}')
    print(f'LOO prediction (partial PC1): r = {r_partial_pc1:.3f}')
    print(f'LOO prediction (partial PC2): r = {r_partial_pc2:.3f}')
    print(f'LOO prediction (partial PC3): r = {r_partial_pc3:.3f}')

    result = {
        'scores': x,
        'spatial_r': y,
        'pc1_alignment': z1,
        'pc2_alignment': z2,
        'pc3_alignment': z3,
        'prediction': r_full,
        'prediction_partial_pc1': r_partial_pc1,
        'prediction_partial_pc2': r_partial_pc2,
        'prediction_partial_pc3': r_partial_pc3
    }

    return result


def map_validation(fc_files, behavior, predictor_map):
    template = brain_template # same MNI template as seed TMS FC map (GSP1000 MNI template)
    predictor_img = nib.load(predictor_map) 
    
    if predictor_img.shape != (91, 109, 91):
        # make sure predictor map is in the same shape
        predictor_img = resample_to_img(predictor_img, template, interpolation='continuous')
    
    # apply affine transformation in case predictor map in an a different MNI template size 

    template_mask = template.get_fdata() > 0 # brain-only voxels from brain template
    predictor_vol = predictor_img.get_fdata() 

    center = np.array(predictor_vol.shape) / 2.0

    def neg_overlap(scale):
        offset = center * (1 - scale)
        scaled = affine_transform(predictor_vol, np.eye(3) / scale, offset=offset, order=1)
        return -np.sum((scaled != 0) & template_mask)

    result_opt = minimize_scalar(neg_overlap, bounds=(0.95, 1.15), method='bounded')
    best_scale = result_opt.x
    offset = center * (1 - best_scale)
    predictor_vol = affine_transform(predictor_vol, np.eye(3) / best_scale, offset=offset, order=1)
    print(f'Affine scale: {best_scale:.4f}, overlap: {np.sum((predictor_vol != 0) & template_mask)}')

    predictor_map_data = predictor_vol[template_mask].flatten()

    # correlate non-zero and non-Nan voxels only

    valid_test = (predictor_map_data != 0) & (~np.isnan(predictor_map_data)) & \
                 (pc1_map != 0) & (~np.isnan(pc1_map)) & \
                 (pc2_map != 0) & (~np.isnan(pc2_map)) & \
                 (pc3_map != 0) & (~np.isnan(pc3_map))

    r_pc1 = np.corrcoef(predictor_map_data[valid_test], pc1_map[valid_test])[0, 1]
    r_pc2 = np.corrcoef(predictor_map_data[valid_test], pc2_map[valid_test])[0, 1]
    r_pc3 = np.corrcoef(predictor_map_data[valid_test], pc3_map[valid_test])[0, 1]
    print(predictor_map)
    print(f'Test map spatial r with PC1: {r_pc1:.3f}, PC2: {r_pc2:.3f}, PC3: {r_pc3:.3f}')

    # same computation as LOO prediction function, expect no leave-one-subject-out scheme is needed

    similarity_to_map = []
    alignment_pc1 = []
    alignment_pc2 = []
    alignment_pc3 = []


    for file in tqdm(fc_files, desc=f'calculating similarity to {predictor_map}'):
        fc_data = na.nifti_getdata(file)
        
        valid = (predictor_map_data != 0) & (~np.isnan(predictor_map_data)) & (fc_data != 0) & (~np.isnan(fc_data)) & (pc1_map != 0) & (~np.isnan(pc1_map)) & (pc2_map != 0) & (~np.isnan(pc2_map)) & (pc3_map != 0) & (~np.isnan(pc3_map))
        
        r = np.corrcoef(fc_data[valid], predictor_map_data[valid])[0, 1]
        similarity_to_map.append(r)

        alignment_pc1.append(np.corrcoef(fc_data[valid], pc1_map[valid])[0, 1])
        alignment_pc2.append(np.corrcoef(fc_data[valid], pc2_map[valid])[0, 1])
        alignment_pc3.append(np.corrcoef(fc_data[valid], pc3_map[valid])[0, 1])

    x = np.array(behavior)
    y = np.arctanh(np.array(similarity_to_map))
    z1 = np.arctanh(np.array(alignment_pc1))
    z2 = np.arctanh(np.array(alignment_pc2))
    z3 = np.arctanh(np.array(alignment_pc3))

    r = pearsonr(x, y)[0]
    r_partial_pc1 = partial_cor(x, y, z1)
    r_partial_pc2 = partial_cor(x, y, z2)
    r_partial_pc3 = partial_cor(x, y, z3)

    result = {'scores': x,
              'spatial_r': y,
              'prediction': r,
              'pc1_alignment': z1,
              'pc2_alignment': z2,
              'pc3_alignment': z3,
              'prediction_partial_pc1': r_partial_pc1,
              'prediction_partial_pc2': r_partial_pc2,
              'prediction_partial_pc3': r_partial_pc3}

    return result

In [ ]:
# ── Load TMS dataset ──────────────────────────────────────────────────────────
tms_dataset = os.path.join(dir, "data/tms_dataset")

tms_fc_files = sorted(glob.glob(os.path.join(tms_dataset, '*AvgR.nii.gz')))
print(f'Found {len(tms_fc_files)} TMS FC files')

# patient metadata included in the repository but available in the supplementary table of Weigand et al. 2018 
df_tms = pd.read_csv(os.path.join(tms_dataset, 'tms_patient_data.csv'))
df_tms = df_tms.sort_values('Patient').reset_index(drop=True)
print(df_tms[['Patient', 'BDI_change_percent']])

bdi_changed = df_tms['BDI_change_percent'].tolist()

# ── Run LOO slnm predictions ───────────────────────────────────────────────────────
tms_loo_prediction   = loo_prediction(tms_fc_files, bdi_changed)

# ── Run control map predictions ───────────────────────────────────────────────────────

# can use any statistical brain map in voxel MNI space
control_maps = sorted(glob.glob(os.path.join(dir, 'data/control_maps/*.nii*')))

print(f'here are the control maps {control_maps}')

results_full = {} # prediction without controlling for PC
results_partial_pc1 = {}
results_partial_pc2 = {}
results_partial_pc3 = {}

for file in control_maps:
    res = map_validation(tms_fc_files, behavior=bdi_changed, predictor_map=file)
    basename = os.path.basename(file)
    results_full[basename] = res['prediction']
    results_partial_pc1[basename] = res['prediction_partial_pc1']
    results_partial_pc2[basename] = res['prediction_partial_pc2']
    results_partial_pc3[basename] = res['prediction_partial_pc3']

results_full['Leave-one-subject-out TMS'] = tms_loo_prediction['prediction']
results_partial_pc1['Leave-one-subject-out TMS'] = tms_loo_prediction['prediction_partial_pc1']
results_partial_pc2['Leave-one-subject-out TMS'] = tms_loo_prediction['prediction_partial_pc2']
results_partial_pc3['Leave-one-subject-out TMS'] = tms_loo_prediction['prediction_partial_pc3']

In [ ]:
# plot TMS prediction for sLNM LOO validation

x = tms_loo_prediction['spatial_r']
y = tms_loo_prediction['scores']
r_loo_tms, p = pearsonr(x, y)
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 16
plt.figure(figsize=(6, 5))
plt.xlabel('Similarity to Leave-One-Subject-Out sLNM Map', fontsize = 16)
plt.ylabel('TMS Response (% BDI reduction)', fontsize = 16)
ax = sns.regplot(
    x=x,
    y=y,
    ci=95,
    line_kws={'color': 'red'})
for collection in ax.collections:
    collection.set_alpha(0.2)

plt.scatter(x, y, s=80, c='red', zorder=2)
plt.yticks([10,30, 50,70 ,90])


plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(28, 11))

results_list = [
    (results_full,        r'$|r|$'),
    (results_partial_pc1, r'$|\mathrm{partial}\ r_{\mathrm{PC1}}|$'),
    (results_partial_pc2, r'$|\mathrm{partial}\ r_{\mathrm{PC2}}|$'),
    (results_partial_pc3, r'$|\mathrm{partial}\ r_{\mathrm{PC3}}|$'),
]

xmax = max(abs(v) for res, _ in results_list for v in res.values()) * 1.1

for ax, (results, xlabel) in zip(axes, results_list):
    sorted_items = sorted(results.items(), key=lambda x: abs(x[1]), reverse=True)
    names = [k.replace('.nii.gz', '').replace('.nii', '') for k, _ in sorted_items]
    values = [abs(v) for _, v in sorted_items]

    # highlight cohort performance as red and control maps as blue 
    colors = ['red' if 'Leave' in k else 'dodgerblue' for k, _ in sorted_items]

    y_pos = range(len(names))
    ax.barh(y_pos, values, color=colors, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names, fontsize=13)
    ax.set_xlabel(xlabel, fontsize=13)
    ax.tick_params(axis='x', labelsize=11)
    ax.set_xlim(0, xmax)
    ax.invert_yaxis()
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()